# Smart Manufacturing Intelligence Platform (SMIP)

# Bronze Layer

## Notebook 01 - Master Data Ingestion

---

### Objective

This notebook ingests all manufacturing master datasets from the Databricks Volume into Bronze Delta tables.

### Source

- Unity Catalog : smip
- Schema : bronze
- Volume : source_data/master_data

### Target

Bronze Delta Tables

- products
- machines
- operators
- operations
- press_programs
- production_halls
- production_lines
- stations
- test_programs
- tools

---

Version

1.2.0

In [0]:
# COMMAND ----------

from framework.core.configuration import *

from framework.core.logger import *

from framework.core.version import *

from framework.io.ingestion import *

from framework.io.delta import *

from framework.quality.validation import *

from framework.quality.metadata import *

from framework.quality.summary import *

In [0]:
# COMMAND ----------

print(f"Catalog        : {CATALOG}")
print(f"Schema         : {BRONZE_SCHEMA}")
print(f"Volume         : {VOLUME}")
print(f"Source Folder  : {MASTER_DATA_FOLDER}")

print()

print("Datasets")

for table, file in MASTER_DATASETS.items():

    print(f"• {table:<22} {file}")

In [0]:
# COMMAND ----------

display(
    dbutils.fs.ls(MASTER_DATA_PATH)
)

In [0]:
# COMMAND ----------

results = []

total_rows = 0

successful = 0

failed = 0

In [0]:
# COMMAND ----------

banner("Starting Bronze Ingestion")

for table_name, file_name in MASTER_DATASETS.items():

    try:

        info(f"Processing {file_name}")

        path = f"{MASTER_DATA_PATH}/{file_name}"

        df = read_csv(path)

        rows = validate_dataframe(df)

        df = add_audit_columns(
            df,
            file_name,
        )

        write_delta(
            df,
            f"{BRONZE_LAYER}.{table_name}",
        )

        success(
            f"{table_name} ({rows} rows)"
        )

        successful += 1

        total_rows += rows

        results.append({

            "Dataset": table_name,

            "Rows": rows,

            "Status": "SUCCESS"

        })

    except Exception as ex:

        failed += 1

        error(
            f"{table_name} : {str(ex)}"
        )

        results.append({

            "Dataset": table_name,

            "Rows": 0,

            "Status": "FAILED"

        })

line()

In [0]:
# COMMAND ----------

summary = spark.createDataFrame(results)

display(summary)

In [0]:
# COMMAND ----------

banner("Execution Summary")

print(f"Datasets Processed : {len(MASTER_DATASETS)}")

print(f"Successful         : {successful}")

print(f"Failed             : {failed}")

print(f"Rows Loaded        : {total_rows:,}")

line()

In [0]:
# COMMAND ----------

display(

    spark.sql(

        f"""
        SHOW TABLES IN {BRONZE_LAYER}
        """

    )

)

In [0]:
# COMMAND ----------

for table in MASTER_DATASETS.keys():

    banner(table.upper())

    display(

        spark.table(

            f"{BRONZE_LAYER}.{table}"

        ).limit(5)

    )